# 歯車式パーセプトロン（犬猫判定機）重み係数計算

Oxford-IIIT Pet Datasetを使い、犬猫の顔画像から特徴量を抽出し、
歯車機構（タレコンギア・反転ギア）に設定する重み係数を計算します。

上から順にセルを実行してください。

In [1]:
# セル1: 必要なライブラリのインストール
!pip install opencv-python-headless scikit-learn numpy pillow -q

In [2]:
# セル2: データセットのダウンロード（Oxford-IIIT Pet Dataset）
!wget -q -O pets.zip "https://codeload.github.com/ml4py/dataset-iiit-pet/zip/refs/heads/master"
!unzip -q -o pets.zip "dataset-iiit-pet-master/images/*.jpg" -d pets_extracted
!unzip -q -o pets.zip "dataset-iiit-pet-master/annotations/list.txt" -d pets_extracted
!unzip -q -o pets.zip "dataset-iiit-pet-master/annotations/xmls/*" -d pets_extracted
print("ダウンロード完了")

ダウンロード完了


In [3]:
# セル3: 顔画像の切り出し（XMLバウンディングボックスを使用）
import xml.etree.ElementTree as ET
from PIL import Image
import numpy as np
import glob
import os

img_dir = "pets_extracted/dataset-iiit-pet-master/images"
xml_dir = "pets_extracted/dataset-iiit-pet-master/annotations/xmls"
list_path = "pets_extracted/dataset-iiit-pet-master/annotations/list.txt"

# SPECIES: 1=猫, 2=犬
species_map = {}
with open(list_path) as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.strip().split()
        if len(parts) >= 3:
            species_map[parts[0]] = int(parts[2])

def crop_face_rgb(xml_path, img_dir, size=(60, 60), margin=0.15):
    """XMLのバウンディングボックスを使って顔部分をRGBで切り出す"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    filename = root.find('filename').text
    bnd = root.find('object/bndbox')
    xmin, ymin = int(bnd.find('xmin').text), int(bnd.find('ymin').text)
    xmax, ymax = int(bnd.find('xmax').text), int(bnd.find('ymax').text)
    img = Image.open(f"{img_dir}/{filename}").convert('RGB')
    w, h = xmax - xmin, ymax - ymin
    mx, my = int(w * margin), int(h * margin)
    left, top = max(0, xmin - mx), max(0, ymin - my)
    right, bottom = min(img.width, xmax + mx), min(img.height, ymax + my)
    face = img.crop((left, top, right, bottom)).resize(size)
    return np.array(face)

xml_files = glob.glob(f"{xml_dir}/*.xml")
cat_faces, dog_faces = [], []
for xml_path in xml_files:
    name = os.path.basename(xml_path).replace('.xml', '')
    species = species_map.get(name)
    if species is None:
        continue
    try:
        face = crop_face_rgb(xml_path, img_dir)
        (cat_faces if species == 1 else dog_faces).append(face)
    except Exception:
        pass

cat_faces = np.array(cat_faces)
dog_faces = np.array(dog_faces)
print(f"猫: {len(cat_faces)}枚, 犬: {len(dog_faces)}枚")

猫: 1181枚, 犬: 2490枚


In [4]:
# セル4: 特徴量抽出（5x5グリッド、エッジ密度+RGB、計100特徴量）
import cv2

def extract_grid_features(img_rgb, grid=5):
    """5x5グリッドで、エッジ密度・R・G・B平均を計算"""
    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)
    edges = cv2.Canny(gray, 50, 150)
    h, w = gray.shape
    features, names = [], []
    for i in range(grid):
        for j in range(grid):
            y0, y1 = i * h // grid, (i + 1) * h // grid
            x0, x1 = j * w // grid, (j + 1) * w // grid
            features.append((edges[y0:y1, x0:x1] > 0).mean())
            names.append(f"edge_{i}_{j}")
            features.append(img_rgb[y0:y1, x0:x1, 0].mean())
            names.append(f"R_{i}_{j}")
            features.append(img_rgb[y0:y1, x0:x1, 1].mean())
            names.append(f"G_{i}_{j}")
            features.append(img_rgb[y0:y1, x0:x1, 2].mean())
            names.append(f"B_{i}_{j}")
    return features, names

cat_feats, dog_feats = [], []
feature_names = None
for img in cat_faces:
    f, feature_names = extract_grid_features(img)
    cat_feats.append(f)
for img in dog_faces:
    f, _ = extract_grid_features(img)
    dog_feats.append(f)

cat_feats = np.array(cat_feats)
dog_feats = np.array(dog_feats)
print(f"特徴量総数: {cat_feats.shape[1]}")

特徴量総数: 100


In [5]:
# セル5: 最適な8特徴量の探索（クロスバリデーションで評価）
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
import random

# データ数を揃える(猫の数に合わせてアンダーサンプリング)
np.random.seed(42)
n_cat = len(cat_feats)
dog_idx = np.random.choice(len(dog_feats), n_cat, replace=False)
dog_feats_b = dog_feats[dog_idx]

X = np.vstack([cat_feats, dog_feats_b])
y = np.array([0] * n_cat + [1] * n_cat)  # 0=猫, 1=犬

scaler = StandardScaler()
X_s = scaler.fit_transform(X)

# 上位25特徴量に絞ってから、8個の組み合わせをランダム探索
selector = SelectKBest(f_classif, k=25)
selector.fit(X_s, y)
top25_idx = list(selector.get_support(indices=True))

random.seed(7)
results = []
N_TRIALS = 2000  # 探索回数。増やすほど良い組み合わせが見つかりやすいが時間がかかる
for _ in range(N_TRIALS):
    combo = tuple(sorted(random.sample(top25_idx, 8)))
    X_c = X_s[:, list(combo)]
    clf = LogisticRegression(max_iter=500)
    scores = cross_val_score(clf, X_c, y, cv=5)
    results.append((scores.mean(), combo))

results.sort(key=lambda x: -x[0])
print("=== 8特徴量、上位5組み合わせ(5-fold CV平均精度) ===")
for score, combo in results[:5]:
    fnames = [feature_names[i] for i in combo]
    print(f"{score:.4f}  {fnames}")

=== 8特徴量、上位5組み合わせ(5-fold CV平均精度) ===
0.7100  ['edge_0_2', 'B_2_2', 'edge_3_0', 'R_3_1', 'edge_3_2', 'R_3_3', 'G_3_3', 'edge_4_2']
0.7092  ['edge_0_2', 'B_2_2', 'G_3_1', 'edge_3_2', 'G_3_3', 'edge_4_1', 'edge_4_3', 'B_4_3']
0.7092  ['edge_0_0', 'edge_0_2', 'B_2_2', 'R_3_1', 'edge_3_2', 'G_3_3', 'R_4_2', 'edge_4_3']
0.7083  ['edge_0_2', 'edge_2_0', 'edge_2_1', 'B_2_2', 'edge_3_2', 'G_3_3', 'R_4_2', 'edge_4_3']
0.7070  ['edge_0_2', 'edge_2_0', 'B_2_2', 'edge_3_0', 'edge_3_2', 'G_3_3', 'edge_4_2', 'G_4_3']


In [6]:
# セル6: 最良の組み合わせで最終的な重み係数を算出
best_score, best_combo = results[0]
best_names = [feature_names[i] for i in best_combo]

X_final = X_s[:, list(best_combo)]
clf_final = LogisticRegression(max_iter=1000)
clf_final.fit(X_final, y)

final_scores = cross_val_score(clf_final, X_final, y, cv=5)
print(f"最終精度: {final_scores.mean():.4f} (±{final_scores.std():.4f})")
print()
print("=== 歯車設計に使う重み係数 ===")
for name, coef in zip(best_names, clf_final.coef_[0]):
    sign = "正(反転なし)" if coef > 0 else "負(反転あり)"
    print(f"  {name:<12}  係数={coef:+.4f}  符号={sign}")

最終精度: 0.7100 (±0.0196)

=== 歯車設計に使う重み係数 ===
  edge_0_2      係数=+0.7722  符号=正(反転なし)
  B_2_2         係数=+0.6864  符号=正(反転なし)
  edge_3_0      係数=+0.0355  符号=正(反転なし)
  R_3_1         係数=-0.3551  符号=負(反転あり)
  edge_3_2      係数=-0.3527  符号=負(反転あり)
  R_3_3         係数=+0.6767  符号=正(反転なし)
  G_3_3         係数=-1.0372  符号=負(反転あり)
  edge_4_2      係数=+0.3034  符号=正(反転なし)


In [7]:
# セル7: 歯車設計用に、重みを0〜2倍の範囲に正規化
coefs = clf_final.coef_[0]
abs_max = np.abs(coefs).max()

print("=== タレコンギア設定値(0〜2倍に正規化) ===")
for name, coef in zip(best_names, coefs):
    normalized = abs(coef) / abs_max * 2.0  # 絶対値が最大のものを2倍に正規化
    sign = "+" if coef > 0 else "-"
    print(f"  {name:<12}  タレコンギア倍率={normalized:.3f}  符号={sign}")

=== タレコンギア設定値(0〜2倍に正規化) ===
  edge_0_2      タレコンギア倍率=1.489  符号=+
  B_2_2         タレコンギア倍率=1.324  符号=+
  edge_3_0      タレコンギア倍率=0.068  符号=+
  R_3_1         タレコンギア倍率=0.685  符号=-
  edge_3_2      タレコンギア倍率=0.680  符号=-
  R_3_3         タレコンギア倍率=1.305  符号=+
  G_3_3         タレコンギア倍率=2.000  符号=-
  edge_4_2      タレコンギア倍率=0.585  符号=+
